# Lesson 05
Run lessons 1–4 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第5课 数据中心的监督学习与量化预测

交付：统一价格/动量/财务/文本特征表、数据质量表、版本化开盘标签、按整日分组且清除重叠的时间切分、仅fit训练集的Pipeline、零收益/Ridge/Logistic/LightGBM基线、固定模型的数据实验、固定数据的模型对照、逐日预测、Rank IC/校准诊断、成本台账、特征组消融和Data Card/Model Card。

核心标签：决策日t收盘已有特征，t+1开盘进入，t+6开盘退出，预测五交易日复权简单收益。label_end记录真实成熟开盘；十日标签是单独登记的数据实验。训练2014—2018、验证2019—2020、开发评估2021—2022；同日所有股票在同一集合，训练标签不能伸入验证，验证标签不能伸入评估。日标签重叠，清除边界重叠并用日期分组汇总，不能把股票行看成独立抽样。

本课使用动量/短期反转、质量、价值与第二课IWM/SPY残差市场状态四类信号；第四课IBM文本作为可用时间对齐的附加特征，其余股票明确缺失，绝不伪造全文覆盖。固定样本的中性化本身使用当天可见横截面；全时序填补、缩放、分位截尾参数都仅从训练学得。

数据实验保持Ridge与评估集不变：注入重复/异常值演示、去重、稳健截尾、完整行丢弃 vs 缺失标记、按日等权、训练内2年半衰期的时间衰减权重、五日 vs 十日标签。注入错误必须标记为人为演示，不声称发现了真实数据错误；改变标签的模型仍在同一个五日评估目标上比较，避免拿不同目标的误差排赢家。

模型实验保持真实干净数据不变：zero、Ridge、Logistic概率、LightGBM。预先登记固定参数，验证窗口用于报告、不用最后一段调参。所有交易模型在共同日期、每5个交易日决策、总多头目标0.8、每腿佣金3bp＋滑点2bp下运行；月内/持仓段每日维护目标权重的费用计入账本。分类概率只用于排序，不冒充收益预测。

数据与模型卡明确存续偏差、滞后股数、镜像版本、词典快照、文本单公司和开发期复用等限制；当前结果不构成独立留出集证明。

参考：[sklearn时间验证](https://scikit-learn.org/stable/modules/cross_validation.html)、[Pipeline防泄漏](https://scikit-learn.org/stable/common_pitfalls.html)、[LightGBM API](https://lightgbm.readthedocs.io/en/stable/Python-API.html)。


In [ ]:
# 05.01 统一特征表、标签版本与数据质量
from sklearn.base import BaseEstimator,TransformerMixin
from lightgbm import LGBMRegressor
F5=PANEL.copy()
for name,mat in {'ret5':CLOSE[ASSETS].pct_change(5,fill_method=None),'ret20':CLOSE[ASSETS].pct_change(20,fill_method=None),
                 'vol20':CLOSE[ASSETS].pct_change(fill_method=None).rolling(20).std()}.items():
    tmp=mat.stack().rename(name).rename_axis(['date','ticker']).reset_index(); F5=F5.merge(tmp,on=['date','ticker'],how='left')
# 第2课IWM/SPY协整残差z作为共同市场状态输入，不冒充每只个股的配对交易信号。
F5['market_pair_z']=F5.date.map(l2_features.z)
F5['pair_fit_last_session']=F5.date.map(l2_features.fit_last_session)
# 文本只有IBM具备来源；按实际ready_at连接，不能按report_end连接。
text_cols=['sentiment','uncertainty','negative','log_length']
F5['text_available_at']=pd.NaT
for col in text_cols: F5['text_'+col]=np.nan
ibm=F5[F5.ticker.eq('IBM')].copy(); ev=E4[['ready_at']+text_cols].sort_values('ready_at').drop_duplicates('ready_at')
joined=pd.merge_asof(ibm[['date','decision_at']].sort_values('decision_at'),ev,left_on='decision_at',right_on='ready_at',direction='backward')
for col in text_cols:
    mapping=pd.Series(joined[col].values,index=joined.date); F5.loc[F5.ticker.eq('IBM'),'text_'+col]=F5.loc[F5.ticker.eq('IBM'),'date'].map(mapping)
text_time=pd.Series(joined.ready_at.values,index=joined.date)
# 统一UTC存储时间字段，避免datetime tz dtype隐式转换。
F5['text_available_at']=pd.to_datetime(F5.date.map(text_time).where(F5.ticker.eq('IBM')),utc=True)
for h in [5,10]:
    labels=forward_open(h).stack().rename('y'+str(h)).rename_axis(['date','ticker']).reset_index()
    F5=F5.merge(labels,on=['date','ticker'],how='left')
    F5['end'+str(h)]=F5.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-(h+1)))
    F5['label_end'+str(h)]=(F5['end'+str(h)]+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
F5['execution_session']=F5.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-1))
F5['execution_at']=(F5.execution_session+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
GROUPS={'price':['ret5','ret20','vol20','momentum_z'],'stat_arb':['market_pair_z'],'fundamental':['quality_z','value_z','size'],
        'text':['text_'+c for c in text_cols]}
FEATURES=sum(GROUPS.values(),[])
F5[FEATURES]=F5[FEATURES].replace([np.inf,-np.inf],np.nan)
F5=F5.dropna(subset=['y5','y10','execution_session']).sort_values(['date','ticker']).reset_index(drop=True)
LABEL_VERSIONS=pd.DataFrame([{'id':'open_h'+str(h)+'_v1','decision':'t close','entry':'t+1 open','exit':'t+'+str(h+1)+' open','horizon':h,'basis':'adjusted simple return','maturity':'label_end'+str(h)} for h in [5,10]])
QUALITY=pd.DataFrame({'missing_rate':F5[FEATURES].isna().mean(),'nonfinite_after_cleanup':(~np.isfinite(F5[FEATURES].fillna(0))).sum(),
 'nunique':F5[FEATURES].nunique(),'source':{c:'price' if c in GROUPS['price'] else 'lesson2 causal residual' if c in GROUPS['stat_arb'] else 'SEC PIT' if c in GROUPS['fundamental'] else 'IBM real filing; others missing' for c in FEATURES}})
FEATURE_REGISTRY=pd.DataFrame([{'feature':f,'family':family,'available_rule':'decision close; past-only source',
 'definition':{'ret5':'5-session past return','ret20':'20-session past return','vol20':'20-session past daily volatility',
 'momentum_z':'12-1 momentum, current cross-sectional neutral residual','market_pair_z':'lesson2 IWM/SPY residual z; prior252-session fit; shared regime',
 'quality_z':'SEC annual ROA neutral residual; filed day-end PIT','value_z':'SEC B/M estimate neutral residual; filed day-end PIT','size':'log lagged-share marketcap estimate'}.get(f,'IBM filing LM feature; latest ready_at<=decision_at; missing other issuers')}
 for family,cols in GROUPS.items() for f in cols])
display(FEATURE_REGISTRY)
print('真实数据质量：duplicates',F5.duplicated(['date','ticker']).sum(),'rows',len(F5),'dates',F5.date.nunique()); display(QUALITY); display(LABEL_VERSIONS)
# 日切分，按10日版本最迟成熟时间清除边界，保证全部标签实验共用同一行集合。
F5['fold']=np.select([F5.date<TRAIN_END,F5.date<VALID_END],['train','validation'],default='development_eval')
F5['purged']=((F5.fold=='train')&(F5.label_end10>=train_boundary))|((F5.fold=='validation')&(F5.label_end10>=valid_boundary))
PURGE_TABLE=F5.groupby('fold').agg(rows=('date','size'),purged=('purged','sum'))
D5=F5[~F5.purged].copy(); TR5=D5[D5.fold.eq('train')]; VA5=D5[D5.fold.eq('validation')]; TE5=D5[D5.fold.eq('development_eval')]
display(PURGE_TABLE); print('05.01 PASS: date-group folds, conservative maturity purge, asof text, versioned labels')


In [ ]:
# 05.02 固定Ridge的数据实验与训练期预处理
class TrainClip(BaseEstimator,TransformerMixin):
    def __init__(self,low=.01,high=.99): self.low=low; self.high=high
    def fit(self,X,y=None):
        arr=np.asarray(X,float); self.lo_=np.nanquantile(arr,self.low,axis=0); self.hi_=np.nanquantile(arr,self.high,axis=0); return self
    def transform(self,X): return np.clip(np.asarray(X,float),self.lo_,self.hi_)
def regression_pipe(cols,model=None,clip=False,indicator=True):
    numeric=[]
    if clip: numeric.append(('clip',TrainClip()))
    numeric.extend([('impute',SimpleImputer(strategy='median',add_indicator=indicator,keep_empty_features=True)),('scale',StandardScaler())])
    return Pipeline([('features',ColumnTransformer([('numeric',Pipeline(numeric),cols)])),('model',Ridge(alpha=10) if model is None else model)])
def prediction_stats(frame,pred,target='y5'):
    z=frame[['date','ticker',target]].copy(); z['prediction']=pred
    ics=[g.prediction.corr(g[target],method='spearman') for _,g in z.groupby('date') if len(g)>=3 and g.prediction.nunique()>1]
    return {'MSE':mean_squared_error(frame[target],pred),'mean_RankIC':np.nanmean(ics) if ics else np.nan,
            'direction_accuracy':np.mean((np.asarray(pred)>0)==(frame[target].to_numpy()>0)),'n':len(frame)}
DUPLICATED=pd.concat([TR5,TR5.iloc[:500]],ignore_index=True) # labeled injected corruption
DEDUP=DUPLICATED.drop_duplicates(['date','ticker'],keep='first')
OUTLIER=TR5.copy(); inject_idx=OUTLIER.index[:100]; OUTLIER.loc[inject_idx,'ret5']=OUTLIER.loc[inject_idx,'ret5']*100
EXPERIMENTS5=[
 {'id':'clean_missing_indicator','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'injected_duplicates','train':DUPLICATED,'clip':False,'indicator':True,'label':'y5','injected':True},
 {'id':'deduplicated','train':DEDUP,'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'injected_outliers','train':OUTLIER,'clip':False,'indicator':True,'label':'y5','injected':True},
 {'id':'outliers_train_clip','train':OUTLIER,'clip':True,'indicator':True,'label':'y5','injected':True},
 {'id':'no_missing_indicator','train':TR5,'clip':False,'indicator':False,'label':'y5','injected':False},
 {'id':'complete_case_train','train':TR5.dropna(subset=FEATURES),'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'date_balanced_weights','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False,'weighted':True},
 {'id':'recency_weights','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False,'weighted':'recency'},
 {'id':'label_h10_to_h5','train':TR5,'clip':False,'indicator':True,'label':'y10','injected':False}]
DATA_RESULTS=[]; DATA_MODELS={}; DATA_PRED={}
for r in EXPERIMENTS5:
    data=r['train']; model=regression_pipe(FEATURES,clip=r['clip'],indicator=r['indicator'])
    kwargs={}
    if r.get('weighted'):
        sw=np.exp(-np.log(2)*(data.date.max()-data.date).dt.days/(2*365.25)) if r.get('weighted')=='recency' else 1/data.date.map(data.groupby('date').size())
        sw=sw/sw.mean(); kwargs={'model__sample_weight':sw}
    model.fit(data,data[r['label']],**kwargs); DATA_MODELS[r['id']]=model
    for fold,frame in [('validation',VA5),('development_eval',TE5)]:
        pred=model.predict(frame)
        if r['label']=='y10': pred=np.sqrt(np.maximum(1+pred,1e-6))-1 # declared constant growth hypothesis for horizon conversion
        DATA_PRED[(r['id'],fold)]=pred
        DATA_RESULTS.append({'experiment':r['id'],'fold':fold,'train_n':len(data),'injected':r['injected'],'train_label':r['label'],**prediction_stats(frame,pred)})
L5_DATA_RESULTS=pd.DataFrame(DATA_RESULTS).set_index(['fold','experiment']); display(L5_DATA_RESULTS)
print('人为异常/重复注入仅在训练副本；评估数据固定。完整行删除主要留下IBM，会改变训练群体，未把它当作推荐清洗。')
print('完整平衡面板的按日等权与逐行等权相同，保留这个无变化结果；另登记训练内2年半衰期的时间衰减权重。')
print('h10转换到h5是假设恒定增长，不保证准确；全部实验保留，无成功筛选。')
print('05.02 PASS: fixed Ridge(alpha=10), date-balanced weights, missing flags, frozen train clipping, registered label experiment')


In [ ]:
# 05.03 固定真实数据的模型对照、逐期预测与特征组消融
from sklearn.dummy import DummyRegressor
LGB_PARAMS={'n_estimators':120,'learning_rate':.05,'num_leaves':15,'max_depth':4,'min_child_samples':100,
            'reg_lambda':10,'random_state':SEED,'n_jobs':2,'verbosity':-1,'deterministic':True,'force_col_wise':True}
MODEL_REGISTRY=pd.DataFrame([
 {'id':'zero','kind':'regression','parameter':'constant=0','training_target':'y5'},
 {'id':'Ridge','kind':'regression','parameter':'alpha=10','training_target':'y5'},
 {'id':'LightGBM','kind':'regression','parameter':json.dumps(LGB_PARAMS),'training_target':'y5'},
 {'id':'Logistic','kind':'classification/ranking','parameter':'C=1; threshold=.5','training_target':'y5>0'}])
L5_MODELS={
 'zero':regression_pipe(FEATURES,DummyRegressor(strategy='constant',constant=0)),
 'Ridge':regression_pipe(FEATURES,Ridge(alpha=10)),
 'LightGBM':regression_pipe(FEATURES,LGBMRegressor(**LGB_PARAMS)),
 'Logistic':regression_pipe(FEATURES,LogisticRegression(C=1,max_iter=2000,solver='liblinear',random_state=SEED))}
L5_PRED=D5[['date','ticker','decision_at','execution_session','execution_at','label_end5','fold','y5']].copy(); MODEL_RESULTS=[]
for name,model in L5_MODELS.items():
    target=(TR5.y5>0).astype(int) if name=='Logistic' else TR5.y5
    model.fit(TR5,target)
    if name=='Logistic': L5_PRED[name]=model.predict_proba(D5)[:,1]
    else: L5_PRED[name]=model.predict(D5)
    for fold,frame in [('validation',VA5),('development_eval',TE5)]:
        if name=='Logistic':
            p=model.predict_proba(frame)[:,1]; y=(frame.y5>0).astype(int)
            ranked=prediction_stats(frame,p); ranked['MSE']=np.nan; ranked['direction_accuracy']=np.nan
            MODEL_RESULTS.append({'model':name,'fold':fold,**ranked,'Brier':brier_score_loss(y,p),'AUC':roc_auc_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p>=.5)})
        else: MODEL_RESULTS.append({'model':name,'fold':fold,**prediction_stats(frame,model.predict(frame))})
L5_MODEL_RESULTS=pd.DataFrame(MODEL_RESULTS).set_index(['fold','model']); display(MODEL_REGISTRY); display(L5_MODEL_RESULTS)
L5_ABLATION_MODELS={}; ABLATION=[]
for name,cols in {'price_only':GROUPS['price'],'no_text':GROUPS['price']+GROUPS['stat_arb']+GROUPS['fundamental'],
                  'no_stat_arb':GROUPS['price']+GROUPS['fundamental']+GROUPS['text'],
                  'no_fundamental':GROUPS['price']+GROUPS['stat_arb']+GROUPS['text'],'fundamental_only':GROUPS['fundamental'],'all':FEATURES}.items():
    model=regression_pipe(cols,Ridge(alpha=10)); model.fit(TR5,TR5.y5); L5_ABLATION_MODELS[name]=model
    for fold,frame in [('validation',VA5),('development_eval',TE5)]: ABLATION.append({'group':name,'fold':fold,**prediction_stats(frame,model.predict(frame))})
L5_ABLATIONS=pd.DataFrame(ABLATION).set_index(['fold','group']); display(L5_ABLATIONS)
print('所有模型与消融均fit相同训练期；逐日预测保留date/ticker/available/entry/maturity，不在评估期重新选参数。')
display(L5_PRED[L5_PRED.fold.eq('development_eval')].head(24))
features_out=L5_MODELS['LightGBM'].named_steps['features'].get_feature_names_out()
IMPORTANCE=pd.Series(L5_MODELS['LightGBM'].named_steps['model'].feature_importances_,index=features_out).sort_values(ascending=False)
display(IMPORTANCE.rename('training_split_importance').to_frame())
fig,ax=plt.subplots(1,2,figsize=(13,4)); dev=L5_PRED[L5_PRED.fold.eq('development_eval')].copy()
for name in ['Ridge','LightGBM','Logistic']:
    daily=dev.groupby('date').apply(lambda g:g[name].corr(g.y5,method='spearman'),include_groups=False)
    daily.rolling(63,min_periods=20).mean().plot(ax=ax[0],label=name)
ax[0].legend(); ax[0].set_title('63-session mean cross-sectional Rank IC')
bins=pd.cut(dev.Logistic,[0,.25,.5,.75,1],include_lowest=True); dev['direction']=(dev.y5>0).astype(int)
cal=dev.groupby(bins,observed=True).agg(prob=('Logistic','mean'),frequency=('direction','mean'),n=('direction','size')); display(cal)
ax[1].plot(cal.prob,cal.frequency,'o-'); ax[1].plot([0,1],[0,1],'k--'); ax[1].set_title('Classification calibration (descriptive, correlated labels)'); plt.tight_layout(); plt.show()
print('05.03 PASS: fixed-data baselines, no random panel split, all experiments preserved')


In [ ]:
# 05.04 统一执行接口、五日决策与成本/回撤比较
DECISION5=pd.DatetimeIndex(sorted(TE5.date.unique()))[::5]
l5_dates=OPEN.loc['2021-01-05':'2022-12-29'].index
L5_SIGNALS=L5_PRED[L5_PRED.date.isin(DECISION5)&L5_PRED.fold.eq('development_eval')].copy()
L5_SIGNALS['model_id']='frozen_2018_baselines'; L5_SIGNALS['horizon']='5 sessions'
assert (L5_SIGNALS.decision_at<L5_SIGNALS.execution_at).all()
PREDICTORS={name:(lambda frame,n=name: L5_MODELS[n].predict_proba(frame)[:,1] if n=='Logistic' else L5_MODELS[n].predict(frame)) for name in L5_MODELS}
for name,m in L5_ABLATION_MODELS.items(): PREDICTORS['ablation_'+name]=lambda frame,m=m:m.predict(frame)
for r in EXPERIMENTS5:
    m=DATA_MODELS[r['id']]; h10=r['label']=='y10'
    PREDICTORS['data_'+r['id']]=lambda frame,m=m,h10=h10:np.sqrt(np.maximum(1+m.predict(frame),1e-6))-1 if h10 else m.predict(frame)
L5_TARGETS={}; L5_LEDGERS={}; L5_TRADES={}
base_rows=TE5[TE5.date.isin(DECISION5)].copy()
for name,predict in PREDICTORS.items():
    scored=base_rows[['date','ticker','execution_session']].copy(); scored['score']=predict(base_rows)
    target=pd.DataFrame(0.,index=l5_dates,columns=ASSETS)
    for date,g in scored.groupby('date'):
        threshold=.5 if name=='Logistic' else 0.
        chosen=g[g.score>threshold].sort_values(['score','ticker']).tail(3)
        if len(chosen)==0: continue
        entry=g.execution_session.iloc[0]; j=OPEN.index.get_loc(entry); window=OPEN.index[j:j+5].intersection(l5_dates)
        target.loc[window,chosen.ticker]=.8/len(chosen)
    L5_TARGETS[name]=target
    L5_LEDGERS[name],L5_TRADES[name]=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
L5_BACKTEST=pd.DataFrame({n:ledger_stats(x) for n,x in L5_LEDGERS.items()}).T
L5_BACKTEST['mean_gross']=[L5_LEDGERS[n].gross_exposure.mean() for n in L5_BACKTEST.index]
SIGNAL_INTERFACE=L5_SIGNALS.melt(id_vars=['date','ticker','decision_at','execution_at','label_end5','execution_session'],
    value_vars=list(L5_MODELS),var_name='model_id',value_name='signal').rename(columns={'ticker':'instrument','decision_at':'available_at'})
SIGNAL_INTERFACE['horizon']='5 trading sessions'
SIGNAL_INTERFACE['signal_type']=np.where(SIGNAL_INTERFACE.model_id.eq('Logistic'),'direction probability','return forecast')
display(SIGNAL_INTERFACE.head(12))
display(L5_SIGNALS.head(18)); display(L5_BACKTEST)
fig,ax=plt.subplots(2,2,figsize=(14,8))
pd.DataFrame({n:L5_LEDGERS[n].wealth for n in L5_MODELS}).plot(ax=ax[0,0],title='Frozen models, net wealth')
pd.DataFrame({n:L5_LEDGERS[n].drawdown for n in L5_MODELS}).plot(ax=ax[0,1],title='Shared-ledger drawdown')
pd.DataFrame({n:x.wealth for n,x in L5_LEDGERS.items() if n.startswith('data_')}).plot(ax=ax[1,0],title='Fixed Ridge data experiments, same 5-session evaluation')
pd.DataFrame({n:x.wealth for n,x in L5_LEDGERS.items() if n.startswith('ablation_')}).plot(ax=ax[1,1],title='Feature-group ablation'); plt.tight_layout(); plt.show()
print('05.04 PASS: all accounts use identical dates, 0.8 gross cap, 3bp commission+2bp slippage, no short borrowing')
print('zero预测不大于0，因此持现金；分类概率仅排序。总收益差异可能来自敞口/换手变化，不等于预测能力优势。')


In [ ]:
# 05.05 数据卡、模型卡、标签/防泄漏/账本验收与课程覆盖
L5_CHECKS=[]
def check5(name,ok):
    assert bool(ok),name
    L5_CHECKS.append({'check':name,'status':'PASS'})
check5('date-ticker keys unique',not D5.duplicated(['date','ticker']).any())
check5('entire date belongs to one fold',D5.groupby('date').fold.nunique().max()==1)
check5('train maximum maturity before validation',TR5.label_end10.max()<train_boundary)
check5('validation maximum maturity before eval',VA5.label_end10.max()<valid_boundary)
check5('lesson2 pair fit strictly past',(D5.pair_fit_last_session<D5.date).all())
check5('registered features equal pipeline inputs',set(FEATURE_REGISTRY.feature)==set(FEATURES))
check5('common signal interface causal',(SIGNAL_INTERFACE.available_at<SIGNAL_INTERFACE.execution_at).all())
check5('strict decision before entry',(D5.decision_at<D5.execution_at).all())
check5('text available before decision',(D5.loc[D5.text_available_at.notna(),'text_available_at']<=D5.loc[D5.text_available_at.notna(),'decision_at'].dt.tz_convert('UTC')).all())
check5('other issuers text correctly missing',D5.loc[~D5.ticker.eq('IBM'),GROUPS['text']].isna().all().all())
check5('all train labels and input prices finite',np.isfinite(TR5[['y5','y10']]).all().all())
check5('injected duplicates explicitly exceed real rows',len(DUPLICATED)>len(TR5) and not TR5.duplicated(['date','ticker']).any())
check5('dedup preserves clean training keys',len(DEDUP)==len(TR5))
# 多个时点手算验证标签，不只镜像shift表达式。
for position in [0,len(TE5)//2,len(TE5)-1]:
    r=TE5.iloc[position]; j=OPEN.index.get_loc(r.date)
    expected5=OPEN.loc[OPEN.index[j+6],r.ticker]/OPEN.loc[OPEN.index[j+1],r.ticker]-1
    expected10=OPEN.loc[OPEN.index[j+11],r.ticker]/OPEN.loc[OPEN.index[j+1],r.ticker]-1
    check5('manual y5 at '+str(position),np.isclose(r.y5,expected5))
    check5('manual y10 at '+str(position),np.isclose(r.y10,expected10))
    check5('maturity timestamp at '+str(position),r.label_end5==OPEN_TIMES[j+6])
# 核验实际已fit填补统计来自训练，未来转换不能改变它。
num=L5_MODELS['Ridge'].named_steps['features'].named_transformers_['numeric']; imp=num.named_steps['impute']
train_median=np.nanmedian(TR5[FEATURES].to_numpy(float),axis=0)
check5('imputer median equals train only',np.allclose(imp.statistics_,train_median,equal_nan=True))
stat_before=imp.statistics_.copy(); future_probe=TE5.copy(); future_probe[FEATURES]=future_probe[FEATURES]*100+100
L5_MODELS['Ridge'].predict(future_probe)
check5('future transformation does not refit',np.array_equal(stat_before,imp.statistics_,equal_nan=True))
clipper=DATA_MODELS['outliers_train_clip'].named_steps['features'].named_transformers_['numeric'].named_steps['clip']
check5('clip bounds fit actual train copy',np.allclose(clipper.lo_,np.nanquantile(OUTLIER[FEATURES],.01,axis=0),equal_nan=True))
# 历史前缀重算价格因子；修改未来价格不会影响过去动量/短期收益定义。
r=TE5.iloc[0]; history=CLOSE.loc[:r.date,r.ticker]
check5('past-only ret5 prefix',np.isclose(r.ret5,history.iloc[-1]/history.iloc[-6]-1))
for name,x in L5_LEDGERS.items():
    check5(name+' common execution window',x.index.equals(l5_dates))
    check5(name+' conserved PnL',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost))
    check5(name+' cost nonnegative',(x.transaction_cost>=0).all())
    check5(name+' long gross constraint',(L5_TARGETS[name]>=0).all().all() and x.gross_exposure.max()<=.800001)
    check5(name+' compound wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
check5('zero baseline remains cash',np.allclose(L5_LEDGERS['zero'].wealth,1))
L5_VALIDATION=pd.DataFrame(L5_CHECKS); display(L5_VALIDATION)
DATA_CARD={'price':PRICE_AUDIT,'universe':ASSETS,'date_rows':len(D5),'unique_dates':D5.date.nunique(),
 'target':'5-session adjusted open-to-open simple return; maturity stored','label_versions':LABEL_VERSIONS.to_dict('records'),
 'filings':{'issuer':'IBM','events':len(E4),'source':'issuer directory linked main filings','dictionary':LM_SOURCE},
 'split':{'train_end_exclusive':str(TRAIN_END.date()),'validation_end_exclusive':str(VALID_END.date()),'purge_uses':'10-session maturity'},
 'limitations':['survivorship','coarse industry','lagged marketcap shares','missing other-issuer text','revised vendor data and SEC contexts','issuer mirror vintage not independently archived','overlapping daily targets; date summaries correlated','development windows already observed']}
MODEL_CARD={'models':MODEL_REGISTRY.to_dict('records'),'frozen_fit_last_decision':str(TR5.date.max()),
 'fit_last_maturity':str(TR5.label_end10.max()),'all_preprocessing':'fit on training only',
 'trade_rule':'top3 positive regression prediction / Logistic p>.5; 0.8 long target; decision every5 sessions; daily weight maintenance',
 'cost':'3bp commission +2bp slippage each leg; adjusted synthetic units; no taxes/market impact',
 'data_experiments':[{k:v for k,v in r.items() if k!='train'} for r in EXPERIMENTS5],
 'not_proven':'independent unseen performance or causal text advantage','checks':len(L5_CHECKS)}
print(json.dumps({'DataCard':DATA_CARD,'ModelCard':MODEL_CARD},ensure_ascii=False,indent=2,default=str))
COVERAGE=pd.DataFrame([
 [3,'FF3/FF5+momentum; reference and same-cost account attribution','03.04/03.06'],[3,'actual momentum/quality/value, filing PIT and neutralization','03.02–03.03'],
 [3,'Rank IC, grouping, equal/trained score, turnover/decay/exposure','03.05–03.06'],
 [4,'real 10-K/10-Q, accession/version/timestamps, full main-PDF text QA','04.01–04.02'],
 [4,'LM sentiment/uncertainty/negation and past market-model events','04.02–04.03'],
 [4,'price/price+dictionary/price+TFIDF train-only comparisons','04.04–04.05'],
 [5,'feature table, quality, h5/h10 labels and maturity purge','05.01'],
 [5,'fixed-model data experiments and fixed-data baselines','05.02–05.03'],
 [5,'dated predictions, calibration/rank, costs/drawdown/ablations/cards','05.03–05.05']],columns=['course','requirement','verified_code'])
display(COVERAGE)
check5('previous lessons validations present',len(L3_VALIDATION)>0 and len(L4_VALIDATION)>0 and len(L2_VALIDATION)>0)
L5_CARD={'course':5,'status':'verified','checks':len(L5_CHECKS),'models':list(L5_MODELS),'accounts':len(L5_LEDGERS)}
print('L5 FINAL PASS',json.dumps(L5_CARD,ensure_ascii=False))
completion=pd.DataFrame({'课程':['第1课','第2课','第3课','第4课','第5课'],
 '验收':['原有代码完成','PASS '+str(len(L2_VALIDATION))+'项','PASS '+str(len(L3_VALIDATION))+'项','PASS '+str(len(L4_VALIDATION))+'项','PASS '+str(len(L5_CHECKS))+'项'],
 '产物':['趋势研究','均值回归/两腿账本','真实财务因子/FF归因','真实申报NLP/事件研究','数据实验/模型对照/统一成本']})
display(Markdown('### 第三、四、五课完整运行验收')); display(completion)
print('LESSONS 03–05 ALL VERIFIED |',pd.Timestamp.now(tz='UTC').isoformat())
# 所有数据、模型、清单与日志留在云端内存和本ipynb输出，无本地导出文件。
